In [ ]:
!pip install torch torchvision torchaudio
!pip install torch-geometric


In [22]:
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import LabelEncoder
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1. B형 노드/엣지 데이터 불러오기
nodes = pd.read_csv("/content/drive/MyDrive/KOPIS/Btype_nodes_utf8sig.csv")
edges = pd.read_csv("/content/drive/MyDrive/KOPIS/Btype_edges_utf8sig.csv")

In [ ]:
# 2. 노드 ID → 인덱스 매핑
node_ids = nodes['node_id'].tolist()
node_id_map = {nid: i for i, nid in enumerate(node_ids)}



In [ ]:
# 3. edge_index 생성
edge_index = torch.tensor([
    [node_id_map[src], node_id_map[tgt]]
    for src, tgt in zip(edges['source_id'], edges['target_id'])
], dtype=torch.long).t().contiguous()



In [ ]:
# 4. 노드 특성 x (node_type → one-hot)
le = LabelEncoder()
node_type_idx = le.fit_transform(nodes['node_type'])
x = torch.tensor(np.eye(len(le.classes_))[node_type_idx], dtype=torch.float)


In [ ]:

# 5. 레이블 y (장르 노드만 레이블, 나머지는 -1)
genres = nodes[nodes['node_type'] == 'GENRE']
genre_ids = genres['node_id'].tolist()
y = torch.full((len(node_ids),), -1, dtype=torch.long)

for idx, row in nodes.iterrows():
    if row['node_type'] == 'GENRE':
        y[idx] = genre_ids.index(row['node_id'])



In [ ]:
# 6. PyG Data 객체 생성
data = Data(x=x, edge_index=edge_index, y=y)

# 7. 저장
torch.save(data, "/content/drive/MyDrive/Btype_gnn_data.pt")


print(data)
print("노드 수:", data.num_nodes)
print("엣지 수:", data.num_edges)
print("특성 차원:", data.num_node_features)

Data(x=[87758, 4], edge_index=[2, 1405574], y=[87758])
노드 수: 87758
엣지 수: 1405574
특성 차원: 4
